Where n = number of observations, k = number of features. Regular R² always increases when more features are added, even if they are irrelevant. Adjusted R² only increases if the new feature genuinely improves the model. It is always ≤ R² and is the preferred metric for multiple linear regression.

---
---

### Q5. Explain the Machine Learning workflow steps from data collection to model evaluation.

-->

Answer:
The standard Machine Learning workflow consists of the following steps in order:

**Step 1 — Data Collection:**
Gather raw data relevant to the problem from sources like databases, APIs, CSV files, web scraping, or surveys. The quality and quantity of data at this stage directly determines the maximum performance a model can achieve.

**Step 2 — Data Exploration (EDA):**
Perform Exploratory Data Analysis to understand the dataset — check shape, data types, summary statistics, missing values, distributions, and correlations. Tools used: df.info(), df.describe(), histograms, heatmaps.

**Step 3 — Data Preprocessing:**
Clean and prepare the data for modeling. This includes handling missing values (fillna, dropna), removing duplicates, encoding categorical variables (Label Encoding, One-Hot Encoding), and detecting/treating outliers.

**Step 4 — Feature Engineering and Selection:**
Create new meaningful features from existing ones (e.g., Family_Size = SibSp + Parch) and select only the features most relevant to the target variable. Remove features with low correlation or high multicollinearity.

**Step 5 — Train-Test Split:**
Divide the dataset into a training set (typically 70–80%) and a testing set (20–30%). The model is trained only on the training set, and performance is evaluated on the unseen test set to simulate real-world prediction.

**Step 6 — Model Selection and Training:**
Choose the appropriate algorithm (Linear Regression, Decision Tree, SVM, etc.) based on the problem type (regression vs classification). Fit the model to the training data using `model.fit(X_train, y_train)`.

**Step 7 — Model Evaluation:**
Evaluate the trained model on the test set using appropriate metrics — RMSE, R², Adjusted R² for regression; Accuracy, Precision, Recall, F1-score for classification. This step reveals how well the model generalises to new data.

**Step 8 — Model Improvement (Tuning):**
If performance is unsatisfactory, improve the model by tuning hyperparameters (GridSearchCV), adding more features, trying different algorithms, or applying regularization (Ridge, Lasso).

---
---

### Q6. Load the Boston Housing dataset and perform basic data exploration using head(), info(), describe().

-->

Answer:
The Boston Housing dataset contains 506 rows and 14 columns — all numerical, with no missing values. The target variable is medv (Median value of owner-occupied homes in $1000s), ranging from $5,000 to $50,000 with a mean of $22,532. Key features include crim (per capita crime rate), rm (average number of rooms), lstat (% lower status population), and tax (property tax rate). Since there are zero missing values across all 14 columns, no imputation is required and the dataset is ready for direct modeling. The feature scales vary widely — for example, crim ranges from 0.006 to 88.97 while nox ranges from 0.38 to 0.87 — so feature scaling will be important before training.

```python
import pandas as pd

df = pd.read_csv('BostonHousing.csv')

print("=== First 5 Rows ===")
print(df.head())

print("\n=== Dataset Info ===")
df.info()

print("\n=== Statistical Summary ===")
print(df.describe())

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Dataset Shape ===")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
```

---
---

### Q7. Perform Exploratory Data Analysis: Create pairplot, correlation heatmap, and identify features highly correlated with medv.

-->

Answer:
The correlation heatmap and pairplot reveal the following features most strongly correlated with medv (house price):

Negative correlations (as these increase, price decreases):
- lstat (% lower status population): -0.738 — strongest negative predictor. Areas with more lower-income residents have lower house prices.
- ptratio (pupil-teacher ratio): -0.508 — higher student-to-teacher ratio (worse schools) = lower prices.
- indus (industrial area %): -0.484 — more industrial zones = lower residential value.
- tax (property tax rate): -0.469 — higher tax burden = lower prices.
- nox (nitric oxide concentration): -0.427 — more pollution = lower prices.

Positive correlations (as these increase, price increases):
- rm (average rooms per dwelling): +0.695 — strongest positive predictor. More rooms = higher price.

Features with correlation above 0.4 (in absolute value) are the most useful for predicting medv and should be prioritized in feature selection.

```python
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('BostonHousing.csv')

# Pairplot (top correlated features only for readability)
key_features = ['rm', 'lstat', 'ptratio', 'nox', 'tax', 'medv']
sns.pairplot(df[key_features], diag_kind='kde', plot_kws={'alpha': 0.4})
plt.suptitle('Pairplot of Key Features vs House Price (medv)', y=1.02)
plt.tight_layout()
plt.show()

# Correlation Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap - Boston Housing Dataset')
plt.tight_layout()
plt.show()

# Correlation with medv
print("Correlation with medv (house price):")
print(df.corr()['medv'].sort_values())
```

---
---

### Q8. Select independent variables (features) and dependent variable (target). Target variable: medv (house price).

-->

Answer:
The target variable (y) is medv — the median value of owner-occupied homes in $1000s. All remaining 13 columns are used as independent variables (X/features): crim, zn, indus, chas, nox, rm, age, dis, rad, tax, ptratio, b, and lstat. All 13 features are retained at this stage since Linear Regression can handle multiple inputs, and excluding features without strong justification can reduce model accuracy. The feature matrix X has shape (506, 13) and the target vector y has shape (506,).

```python
X = df.drop('medv', axis=1)
y = df['medv']

print("Feature matrix shape (X):", X.shape)
print("Target vector shape (y) :", y.shape)
print("\nFeatures used:\n", X.columns.tolist())
print("\nTarget variable: medv")
print(y.describe())
```

---
---

### Q9. Split the dataset into training and testing sets (80% training, 20% testing).

-->

Answer:
The dataset of 506 records was split into 404 training records (80%) and 102 testing records (20%) using random_state=42 to ensure reproducibility — meaning the same split is produced every time the code is run. The training set is used to teach the model the relationship between features and house prices. The test set is kept completely unseen during training and is used only to evaluate how well the model generalises to new data. This separation prevents overfitting evaluation — a model that memorises training data might score 100% on training but fail on new data.

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set size :", X_train.shape)
print("Testing set size  :", X_test.shape)
print("\nTraining rows:", len(X_train), "| Testing rows:", len(X_test))
```

---
---

### Q10. Train a Linear Regression model, make predictions, calculate all metrics, write the equation, and interpret coefficients.

-->

Answer:
Model Training and Predictions: The Linear Regression model was trained on 404 training records and predictions were made on the 102 test records.

Evaluation Metrics:
- MAE (Mean Absolute Error) : 3.189 — on average, predictions are off by $3,189
- MSE (Mean Squared Error) : 24.291 — average squared error (penalizes large errors more)
- RMSE (Root Mean Sq. Error) : 4.929 — typical prediction error is about $4,929
- R² Score : 0.669 — the model explains 66.9% of variance in house prices
- Adjusted R² : 0.620 — after penalizing for 13 features, adjusted fit is 62.0%

The R² of 0.669 means the model captures about two-thirds of the price variation — a moderate fit. The gap between R² and Adjusted R² (0.669 vs 0.620) suggests some features may not be contributing meaningfully and could be dropped to simplify the model.

Linear Regression Equation:
medv = 30.25 − 0.113(crim) + 0.047(zn) + 0.021(indus) + 2.687(chas) − 17.767(nox) + 3.810(rm) + 0.001(age) − 1.476(dis) + 0.306(rad) − 0.012(tax) − 0.953(ptratio) + 0.009(b) − 0.525(lstat)

Interpretation of Coefficients:
- rm (+3.810): Strongest positive driver — each additional room increases predicted price by about $3,810, holding other features constant. Matches the strong positive correlation (+0.695) from EDA.
- nox (−17.767): Largest negative coefficient — a one-unit rise in nitric oxide concentration lowers predicted price by $17,767, confirming pollution strongly hurts value.
- lstat (−0.525): Each 1% increase in lower-status population decreases predicted price by about $525, consistent with its strong negative correlation (−0.738).
- dis (−1.476): Greater distance to employment centers is associated with lower prices in this multivariate model, likely due to interaction with features like crim and indus.
- ptratio (−0.953): Each one-unit increase in pupil-teacher ratio decreases price by about $953, aligning with the negative correlation found in EDA.
- chas (+2.687): Homes bordering the Charles River are worth about $2,687 more than similar homes that don't.
- crim, tax, indus, age, b, zn, rad: These have relatively small coefficients, contributing less individually once stronger features are accounted for — consistent with the gap between R² and Adjusted R².

Summary: Room count (rm), pollution (nox), and socioeconomic status (lstat) are the dominant factors driving predicted house prices, while age, b, and zn contribute minimally and could be candidates for removal in a simplified model.

---
---